In [1]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

d:\Ai Data Engineering\RAG_POC\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:

# 0 Initialize the SAME embedding model used in the first notebook
# This is required so Chroma can understand the mathematical vectors
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")

# Point to your storage folder
persist_directory = "../chroma_db"

# Define the 'vectorstore' variable by loading the folder
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)

print(f"Success: vectorstore is now defined. Found {len(vectorstore.get()['ids'])} chunks.")

Success: vectorstore is now defined. Found 438 chunks.


In [3]:
# 1. Initialize the Local Docker Model
# Docker Model Runner serves an OpenAI-compatible API on port 12434
llm = ChatOpenAI(
    model="ai/qwen3:0.6B-F16",
    base_url="http://localhost:12434/engines/v1",
    api_key="not-needed", 
    temperature=0
)

In [4]:
# 2. Setup the Chain (Use your existing prompt logic)
system_prompt = (
    "You are a SQL expert. Use the following context to answer the question. "
    "Include code examples. If the answer isn't in the context, say so."
    "\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [5]:
# 3. Create the Chain (Using the same vectorstore you already built)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("Local Qwen3 model is now powering your SQL Bot!")

Local Qwen3 model is now powering your SQL Bot!


In [7]:
# Quick test to check the local model response
test_query = "hi, give me update query for customer table where cust_id is 22 and cust name i want to update from sumit to swapnil "
try:
    response = rag_chain.invoke({"input": test_query})
    print("Local Model Response:")
    print(response["answer"])
except Exception as e:
    print(f"Connection Error: Ensure your Docker container for {llm.model} is running at {llm.base_url}")
    print(f"Details: {e}")

Local Model Response:
Here is the SQL UPDATE query to update the customer record with ID 22 from "sumit" to "swapnil":

```sql
UPDATE CUSTOMERS 
SET NAME = 'swapnil' 
WHERE ID = 22;
```

This query uses the `UPDATE` statement with the `SET` clause to modify the `NAME` column and the `WHERE` clause to specify the customer ID. The `WHERE` clause ensures only the record with ID 22 is updated.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create UI Elements
search_bar = widgets.Text(
    placeholder='Enter your SQL question...',
    description='Query:',
    layout=widgets.Layout(width='75%')
)
ask_button = widgets.Button(
    description='Run Query',
    button_style='primary'
)
output_area = widgets.Output()

def process_query(b):
    with output_area:
        user_input = search_bar.value
        if not user_input.strip():
            return
        
        clear_output()
        print(f"Processing query: {user_input}")
        
        try:
            # Execute the RAG chain using the local model
            response = rag_chain.invoke({"input": user_input})
            
            print("\nAnalysis Results:")
            print(response['answer'])
            
            # Extract metadata for transparency
            pages = set([doc.metadata.get('page', 'Unknown') for doc in response['context']])
            print(f"\nDocument Reference: Page(s) {', '.join(map(str, pages))}")
            
        except Exception as e:
            print(f"Error during execution: {e}")

# Map triggers
ask_button.on_click(process_query)
search_bar.on_submit(process_query)

# Display Interface
print("SQL Knowledge Bot - Local Execution Mode")
display(widgets.HBox([search_bar, ask_button]), output_area)

SQL Knowledge Bot - Local Execution Mode


C:\Users\Swapn\AppData\Local\Temp\ipykernel_4924\2653788059.py:41: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  search_bar.on_submit(process_query)


Output()